# SV + TRGT explorer

Interactive **Plotly** view of structural variants (Kanpig-style phased BCF) and **TRGT** repeat calls for one sample. Only SV sites with a **called genotype** (non-missing `GT`) are drawn.

**Locus:** point at **indexed** whole-genome BCF/VCF if you like — `pysam` only **fetches the interval** you set (e.g. `chr6:31803187-32050925`), so you are not loading the whole genome into memory.

**Setup:** plotting implementation lives in `notebooks/visualize_plot.py`. From the repo root, `pip install -r dev-requirements.txt`, then open this notebook. **TRGT/SV** variant bars use true genomic width (`max(100 bp, alternate-allele length)`); **`MIN_SEGMENT_PX`** only widens **UCSC** annotation tracks at draw time so short repeats stay visible. Zooming does not recompute widths (avoids `FigureWidget` / ipywidgets sync issues). For Jupyter Classic, `notebook_connected` works when `ipywidgets` is installed.

**Allele sequence strips:** concrete TRGT consensus alleles and SV insertion sequences draw as one-bp-wide IGV-colored bars in genomic coordinates below each variant. Each variant toggles independently: first click shows its strip, second click hides it; several strips can be visible at once (requires the `FigureWidget` layout).

In [1]:
"""Load plotting helpers from ``visualize_plot.py`` (same folder as this notebook)."""
from __future__ import annotations

import sys
from pathlib import Path

# Ensure ``notebooks/`` is on ``sys.path`` whether the kernel cwd is repo root or ``notebooks/``.
_here = Path.cwd().resolve()
for _p in (_here / "notebooks", _here, _here.parent / "notebooks"):
    if (_p / "visualize_plot.py").exists():
        sys.path.insert(0, str(_p))
        break

from visualize_plot import (
    build_figure,
    display_figure_with_genomic_cursor,
    find_data_dir,
    parse_locus,
)


In [ ]:
# --- Configure paths, locus, and sample (edit here; whole-genome files OK if indexed) ---
from IPython.display import display

DATA_DIR = find_data_dir()
SV_PATH = DATA_DIR / "1000920.phase2.bcf"
TRGT_PATH = DATA_DIR / "1000920_trgt.TR_Explorer_1.01.chr6.31803187_32050925.norm.vcf.gz"

LOCUS = "chr6:31803187-32050925"  # 1-based inclusive; only this interval is fetched
SAMPLE = None  # None → first sample in SV BCF; or e.g. "1000920"
GENOME = "hg38"  # UCSC DB name for RepeatMasker / simpleRepeat / segDup
UCSC_TRACKS = True  # needs network access to genome-mysql.soe.ucsc.edu
MIN_ALLELE_BP = 20  # TRGT: only draw haps with |alt - ref| bp >= this (AL vs ref span; 0 = off)
MIN_SEGMENT_PX = 6.0  # min on-screen width for TRGT/SV/UCSC spans at default view (baked at build time; use 0 for exact geometry)

CHROM, WIN_START, WIN_END = parse_locus(LOCUS)

assert SV_PATH.exists(), f"Missing SV BCF: {SV_PATH}"
assert TRGT_PATH.exists(), f"Missing TRGT VCF: {TRGT_PATH}"

fig = build_figure(
    SV_PATH,
    TRGT_PATH,
    CHROM,
    WIN_START,
    WIN_END,
    sample=SAMPLE,
    genome=GENOME,
    ucsc_tracks=UCSC_TRACKS,
    min_allele_bp=MIN_ALLELE_BP,
    min_segment_px=MIN_SEGMENT_PX,
)
out = display_figure_with_genomic_cursor(fig, CHROM, WIN_START, WIN_END)
if out is not None:
    display(out)
